# Deforestation and forest-capacity loss analysis

## Purpose and scope

This notebook inspects reported deforestation, the annual forest-biomass addition capacity associated with affected area, and the relative erosion summaries prepared for the current results. It keeps the physical capacity-loss measure distinct from emissions released from carbon already stored in forest biomass.

The analysis is limited by the reporting coverage shown in the data. A missing report is not interpreted as zero deforestation. The notebook reads the current tables and does not automatically extrapolate non-reporting units.

## 1. Set up and load the current deforestation outputs

The country-level file contains the reporting flag, mean lost capacity, and erosion relative to GCW. The manuscript table summarizes erosion by group; supplementary files preserve reported trends and scenario assumptions.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise FileNotFoundError('Run this notebook from within the repository.')

DATA = ROOT / 'data' / 'derived'
RESULTS = ROOT / 'results'
TABLES = RESULTS / 'tables' / 'manuscript'
VALIDATION = RESULTS / 'validation'
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)
print(f'Repository: {ROOT}')

Repository: C:\Users\aurel\Desktop\M2\Stage ecoseas-csm\CSM\BNC2 paper\repo


## 2. Count reporters and positive-loss cases

The following counts separate units with a deforestation report from units with a positive reported loss. This distinction matters because the denominator of a group comparison can change depending on reporting coverage and on whether zero or negative entries are retained.

In [2]:
accounts = pd.read_csv(TABLES / 'Supplementary_country_level_accounts_2024.csv')
reporter = accounts['reported_deforestation'].fillna(False).astype(bool)
loss = pd.to_numeric(accounts['loss_tco2_mean'], errors='coerce')
print({'all_unit_records':len(accounts), 'reporting_units':int(reporter.sum()), 'positive_loss_units':int((reporter & loss.gt(0)).sum()), 'reported_nonpositive_or_zero':int((reporter & loss.le(0)).sum())})
coverage = accounts.loc[reporter, ['UNIT_CODE', 'name', 'groups', 'reported_deforestation', 'loss_tco2_mean', 'erosion_pct_of_gcw', 'loss_value_usd2015_yr_mean']]
display(coverage.head(20))

{'all_unit_records': 267, 'reporting_units': 139, 'positive_loss_units': 105, 'reported_nonpositive_or_zero': 34}


,UNIT_CODE,name,groups,reported_deforestation,loss_tco2_mean,erosion_pct_of_gcw,loss_value_usd2015_yr_mean
3,AGO,Angola,LDCs,True,1.748833e+06,0.806862,9.785995e+08
8,ARG,Argentina,Developing economies,True,9.058507e+05,0.502049,5.068895e+08
20,AUS,Australia,Developed economies,True,2.162169e+06,0.332948,1.209891e+09
21,AUT,Austria,Developed economies,True,2.689959e+04,0.131407,1.505228e+07
22,AZE,Azerbaijan,Developing economies,True,4.439168e+05,7.060462,2.484038e+08
26,BEL,Belgium,Developed economies,True,1.122127e+04,0.319155,6.279120e+06
27,BEN,Benin,LDCs,True,3.760347e+05,3.502890,2.104188e+08
29,BFA,Burkina Faso,LDCs,True,4.416793e+05,3.982583,2.471518e+08
30,BGD,Bangladesh,LDCs,True,6.232127e+04,1.284468,3.487329e+07
31,BGR,Bulgaria,Developed economies,True,0.000000e+00,0.000000,0.000000e+00


## 3. Compare erosion by economic group

Table 3 contains the current group-level erosion summary. Check the included sample sizes and summary statistic definitions in the table before comparing groups. The purpose of this cell is to expose the calculated values and their denominators, not to select a narrative interpretation.

In [3]:
table3 = pd.read_csv(TABLES / 'Table3_forest_capacity_erosion.csv')
display(table3)
by_group = accounts.loc[reporter].groupby('groups', dropna=False).agg(reporting_units=('UNIT_CODE','size'), positive_loss_units=('loss_tco2_mean',lambda x: int(pd.to_numeric(x,errors='coerce').gt(0).sum())), median_capacity_loss_tco2_yr=('loss_tco2_mean',lambda x: pd.to_numeric(x,errors='coerce').median()), median_erosion_pct=('erosion_pct_of_gcw',lambda x: pd.to_numeric(x,errors='coerce').median())).reset_index()
display(by_group)

,group,reporting_units,positive_loss_units,lost_capacity_mtco2_yr,lost_gcw_billion_usd2015_yr,median_erosion_pct_of_gcw
0,Developed economies,47,32,3.125620,1.749012,0.042338
1,Developing economies,51,37,20.548622,11.498452,0.241385
2,LDCs,24,22,11.283131,6.313734,1.252482
3,SIDS,17,14,0.308763,0.172775,0.100331
4,Unclassified,0,0,NaN,NaN,NaN
5,Total,139,105,35.266135,19.733973,NaN


,groups,reporting_units,positive_loss_units,median_capacity_loss_tco2_yr,median_erosion_pct
0,Developed economies,47,32,2775.552933,0.042338
1,Developing economies,51,37,24890.821140,0.241385
2,LDCs,24,22,220308.606550,1.252482
3,SIDS,17,14,1695.933443,0.100331


## 4. Inspect reported trends and illustrative scenarios

The trend table preserves the source intervals and annual reported areas. The scenario table records the assumptions used for projections or capitalisation. Values from scenarios should be kept distinct from observed annual losses.

In [4]:
trend = pd.read_csv(TABLES / 'Supplementary_deforestation_trend.csv')
scenarios = pd.read_csv(TABLES / 'Supplementary_deforestation_scenarios.csv')
display(trend)
display(scenarios)

,fra_interval,reported_deforestation_kha_per_year,midpoint_year,reported_deforestation_mha_per_year
0,1990-2000,15818.77,1995.0,15.81877
1,2000-2010,13328.62,2005.0,13.32862
2,2010-2015,9878.10,2012.5,9.87810
3,2015-2020,9477.81,2017.5,9.47781
4,2020-2025,8484.25,2022.5,8.48425


,metric,value,unit,assumption_or_formula,discount_rate,ibrahim_allocation_rate
0,Current annual lost forest addition capacity,35.266135,MtCO2/yr,Sum of reported-unit estimates; 139 reporting ...,NaN,NaN
1,Ten-year cumulative capacity not supplied,0.352661,GtCO2 over 10 years,Current annual loss repeats for 10 years; no r...,NaN,NaN
2,Ten-year cumulative loss relative to current a...,2.388684,% of current annual forest additions,Ten-year cumulative loss / global annual fores...,NaN,NaN
3,Current annual lost GCW value,19.733973,billion US$2015/yr,Sum of reported-unit annual lost GCW estimates,NaN,NaN
4,Perpetuity value of annual lost GCW at 2% disc...,0.986699,trillion US$2015,Annual lost GCW value / 0.02; perpetual consta...,0.02,NaN


## 5. Unit-level cases for follow-up

Rank the positive capacity-loss cases to inspect the observations driving upper tails. The table retains the original unit identifiers so any manuscript statement can be traced to the source export and reporting coverage.

In [5]:
positive = accounts.loc[reporter & loss.gt(0), ['UNIT_CODE', 'name', 'region', 'groups', 'loss_tco2_mean', 'erosion_pct_of_gcw', 'loss_value_usd2015_yr_mean']].copy()
display(positive.sort_values('loss_tco2_mean', ascending=False).head(20))

,UNIT_CODE,name,region,groups,loss_tco2_mean,erosion_pct_of_gcw,loss_value_usd2015_yr_mean
40,BRA,Brazil,Americas,Developing economies,7.860908e+06,0.468763,4.398751e+09
115,IND,India,Asia,Developing economies,2.509565e+06,0.920745,1.404284e+09
54,COD,Democratic Republic of the Congo,Africa,LDCs,2.210384e+06,0.463510,1.236871e+09
20,AUS,Australia,Oceania,Developed economies,2.162169e+06,0.332948,1.209891e+09
113,IDN,Indonesia,Asia,Developing economies,1.853758e+06,0.563790,1.037313e+09
3,AGO,Angola,Africa,LDCs,1.748833e+06,0.806862,9.785995e+08
51,CHN,"China (incl. Taiwan, Hong Kong SAR and Macao SAR)",Asia,Developing economies,1.593247e+06,0.146122,8.915378e+08
241,TZA,United Republic of Tanzania,Africa,LDCs,1.493792e+06,1.008281,8.358854e+08
194,PRY,Paraguay,Americas,Developing economies,1.053860e+06,2.151407,5.897114e+08
205,SDN,Sudan,Africa,LDCs,9.191274e+05,1.220496,5.143188e+08
